# Week 1 Day 4 Lab - Open-Source, Local, Token Cost (CPU version)

## Small models on UnderCover's claims

We send 20 of the claims in the queue to free models. `openrouter/free` picks a free model on OpenRouter for each request, and three small open-weight models run on your own computer's CPU with Ollama. Every model gets a smaller prompt than on Day 2. Then we put accuracy, speed and memory side by side.

## 4 parts:

PART 1: The data and a smaller prompt

PART 2: Free models on OpenRouter

PART 3: Three small models on your computer with Ollama

PART 4: Accuracy, speed and memory side by side

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This CPU version of the lab is for a computer without a GPU. It uses <code>openrouter/free</code>, which needs no credit, and three models small enough to run on the CPU. Each has 1 billion parameters or fewer. Models this small can give some odd answers.<br/><br/>
            The full version with six paid models is <a href="week1day4.ipynb">week1day4.ipynb</a>. The free version with larger local models is <a href="week1day4free.ipynb">week1day4free.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit, and this lab makes about 22 of them, a few more when a busy model means asking again. The local models have no limit.<br/><br/>
            Part 3 needs Ollama and about 2.5 GB of disk for the three models.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import APIError, LengthFinishReasonError, OpenAI
from pydantic import BaseModel, Field, ValidationError
from tqdm.auto import tqdm

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


## Part 1: The data and a smaller prompt

The seed data is in the repo's `data` folder, as on Day 2. Today we leave out the 5 manual claims. They depend on the claims manual or the customer's history, which the agent gets on Day 5, so the policy wording alone can't settle them. That leaves 20 claims. Some models take several minutes on all 20, so set `TINY = True` for a quicker run on 4 of them.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [4]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim["claim_id"]]["difficulty"] != "manual"]
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

print(f"{len(claims)} claims")

4 claims


In [5]:
decisions = pd.DataFrame([labels[claim["claim_id"]] for claim in claims])
decisions["decision"].value_counts()

decision
approve    2
decline    1
refer      1
Name: count, dtype: int64

12 of the 20 are approvals, so a model that approves every claim would get 60% of the decisions right. Keep that number in mind when we look at the results.

### A smaller prompt

On Day 2 the prompt had the whole policy wording, and the policy record and the claim as indented JSON. Today we trim it in three ways:

1. A short version of the policy wording, in `data/policy_wording_short.md`
2. Only the sections of it that matter for the type of claim
3. The policy record and the claim as compact JSON, without the ids and without the text of the documents

A smaller prompt costs less, runs faster and suits a small model on your own computer. It can also leave out something the model needs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: tokens</h2>
            <span style="color:#118AB2;">Models read and write tokens, which are pieces of words. In English a token is about three quarters of a word.<br/><br/>
            Providers charge per million tokens, with one price for the tokens you send and another for the tokens the model writes. The tokens it writes usually cost several times more, and reasoning tokens count as written.
            </span>
        </td>
    </tr>
</table>

In [6]:
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")

print(f"The policy wording is {len(policy_wording.split()):,} words")
print(f"The short version is {len(short_wording.split()):,} words")

The policy wording is 1,369 words
The short version is 808 words


The short wording has a section for each benefit, with headings that start with `## `. We split it into sections so we can pick the ones each claim needs.

In [7]:
def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections


opening, sections = split_sections(short_wording)
list(sections)

['Limits',
 'Cancellation and Interruption',
 'Trip Delay',
 'Baggage Delay',
 'Baggage',
 'Passport',
 'Emergency Medical',
 'Add-ons',
 'Never covered',
 'Payment',
 'Claims']

In [8]:
BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}


def make_wording(claim):
    """The sections of the short wording that this claim needs: the limits, its benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    return "\n\n".join([opening.strip()] + [sections[heading].strip() for heading in headings])

In [9]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


def make_user_prompt(claim):
    """The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim["policy_id"]]
    record = {key: value for key, value in policy.items() if key not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": document["name"], "type": document["type"]} for document in claim["documents"]]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording it needs, then the policy record and the claim."""
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{make_wording(claim)}\n"},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

Here's the whole prompt for the first claim, exactly as the model sees it.

In [10]:
messages = make_messages(claims[0])
print(messages[0]["content"])
print(messages[1]["content"])

You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

# UnderCover Travel Insurance: Policy Wording, short version

All amounts are in US dollars. The policy record shows the plan, the add-ons, the trip dates, the trip cost and the travelers.

## Limits

| Benefit | Economy | Premium | First |
|---|---|---|---|
| Cancellation and Interruption maximum | 1,500 | 5,000 | 10,000 |
| Trip Delay waiting period, hours | 12 | 6 | 3 |
| Trip Delay maximum | 300 | 800 | 1,500 |
| Baggage Delay waiting period, hours | 24 | 12 | 6 |
| Bag

## Part 2: Free models on OpenRouter

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Free models on OpenRouter</h2>
            <span style="color:#118AB2;">Free models on OpenRouter have an id that ends in <code>:free</code>, and most of them are open-weight. <code>openrouter/free</code> picks one of them for each request and moves on to another when one is busy. Our results record which model answered.<br/><br/>
            With <code>require_parameters</code> OpenRouter only picks models that support every parameter we send, including Structured Outputs. We leave the reasoning level at the model's default, since setting it made free models hang on Day 2.
            </span>
        </td>
    </tr>
</table>

We use the same Pydantic class as on Day 2 for Structured Outputs. The rationale comes before the decision so the model works through the claim before it decides.

In [11]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

### Calling the LLM

`decide` is Day 2's function with two more things in the result: the model that answered and the number of input tokens.

Free models are busy at times. A request can come back with a rate limit error, or with an error in place of an answer, which the OpenAI SDK reports as a `TypeError`. Now and then a reply comes back empty, or with JSON that doesn't fit `ClaimDecision`, which Pydantic reports as a `ValidationError`.

The free models reason before they answer, and a reply usually takes 1,000 to 3,000 tokens. Once in a while a small one keeps on writing. `max_tokens=8000` cuts it off, and the OpenAI SDK raises a `LengthFinishReasonError`.

In each case `ask` tries again, up to three times, and `openrouter/free` picks another free model when it does.

In [12]:
def ask(model, messages):
    """Send the messages to a free model on OpenRouter with Structured Outputs, and ask again if there's no answer."""
    for attempt in range(1, 4):
        try:
            response = openrouter.chat.completions.parse(
                model=model,
                messages=messages,
                response_format=ClaimDecision,
                max_tokens=8000,
                extra_body={"provider": {"require_parameters": True}},
            )
            if response.choices[0].message.parsed:
                return response
            print(f"An empty reply on try {attempt}, so we ask again")
        except (APIError, LengthFinishReasonError, TypeError, ValidationError) as error:
            print(f"{type(error).__name__} on try {attempt}, so we ask again")
    raise RuntimeError(f"No answer from {model} after 3 tries")


def decide(claim, model):
    """Ask a free model on OpenRouter to decide one claim."""
    start = time.time()
    response = ask(model, make_messages(claim))
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "where": "OpenRouter",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "answered_by": response.model,
        "input_tokens": response.usage.prompt_tokens,
        "seconds": round(time.time() - start, 1),
    }

In [13]:
# Skipped in this run: OpenRouter free-tier daily quota was exhausted.


### What the smaller prompt saves

Here's the same claim with Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON. The two requests may go to different free models, which count tokens a little differently.

In [14]:
# Skipped in this run: OpenRouter free-tier daily quota was exhausted.


### Now running all 20 claims

This time we record whether the model gets the decision right, and whether it gets the decision and the payout right.

In [15]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def score(answer, label):
    """Add the difficulty, and whether the decision, and the decision with the payout, match the handler's."""
    return {
        **answer,
        "difficulty": label["difficulty"],
        "decision_right": answer["decision"] == label["decision"],
        "right": is_right(answer, label),
    }


def report(results):
    """Print how a run did: the decisions right, the decisions right with the payout too, and the time."""
    decisions_right = sum(result["decision_right"] for result in results)
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, in {seconds:.0f} seconds")

In [16]:
def run(model, claims, labels):
    """Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

In [17]:
# Skipped in this run: OpenRouter free-tier daily quota was exhausted.


These are the free models that answered, with the number of claims each one decided.

In [18]:
# Skipped in this run: OpenRouter free-tier daily quota was exhausted.


## Part 3: Three small models on your computer with Ollama

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Ollama</h2>
            <span style="color:#118AB2;">Ollama runs open-weight models on your own computer. Install it from <a href="https://ollama.com">ollama.com</a> and start it. It runs in the background and serves models on your computer. The <code>ollama</code> Python package talks to it.<br/><br/>
            Models are downloaded with a pull. Ollama loads a model into memory when you first use it and unloads it after a few minutes without use.
            </span>
        </td>
    </tr>
</table>

Without a GPU, Ollama runs a model on the CPU. That's much slower, so we use three models with 1 billion parameters or fewer:

- `qwen3.5:0.8b` is Qwen 3.5 with 0.8 billion parameters, a 1.0 GB download, from the same family as the 4B model in the full version
- `gemma3:1b` is Gemma 3 with 1 billion parameters, an 815 MB download. Gemma 4 has no model this small
- `granite4:350m` is IBM's Granite 4 with 350 million parameters, a 708 MB download

The pulls take a minute or two the first time.

In [19]:
print(ollama.pull("qwen3.5:0.8b").status)
print(ollama.pull("gemma3:1b").status)
print(ollama.pull("granite4:350m").status)

success


success


success


Ollama takes our Pydantic schema as `format` and holds the model's reply to it, so even a small model answers in JSON. `think=False` turns reasoning off for Qwen, and `num_ctx` sets the context window, which Ollama otherwise sets to 4,096 tokens whatever the model supports.

`num_predict` stops the reply after 300 tokens. A decision with 2 or 3 sentences of rationale takes about 100 tokens. Sometimes a small model keeps on writing. When the reply is cut off, the JSON is unfinished. We record that as "no answer", which counts as wrong. Ollama also stops a model that repeats itself over and over, with a `ResponseError`. That counts as no answer too.

We also record how much memory Ollama is using for the model, which decides whether it fits on your computer.

In [20]:
def memory_in_use(model):
    """The memory Ollama is using for a model it has loaded, in GB."""
    for running in ollama.ps().models:
        if running.model == model:
            return round(running.size / 1e9, 1)
    return None

In [21]:
def decide_locally(claim, model):
    """Ask a model on this computer to decide one claim, through Ollama, with the reply cut off at 300 tokens."""
    start = time.time()
    try:
        response = ollama.chat(
            model=model,
            messages=make_messages(claim),
            format=ClaimDecision.model_json_schema(),
            think=False,
            options={"num_ctx": 8192, "num_predict": 300},
        )
        answer = ClaimDecision.model_validate_json(response.message.content)
        decision, payout, rationale = answer.decision, answer.payout, answer.rationale
    except (ollama.ResponseError, ValidationError) as error:
        print(f"No answer on {claim['claim_id']}: {type(error).__name__}")
        decision, payout, rationale = "no answer", 0.0, str(error)
    return {
        "model": model,
        "where": "This computer",
        "claim_id": claim["claim_id"],
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
        "memory": memory_in_use(model),
        "seconds": round(time.time() - start, 1),
    }

In [22]:
def run_locally(model, claims, labels):
    """Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim["claim_id"]]))
    ollama.generate(model=model, keep_alive=0)  # unload the model, so the next one has the memory
    report(results)
    return results

In [23]:
first_local = decide_locally(claims[0], "qwen3.5:0.8b")
first_local

{'model': 'qwen3.5:0.8b',
 'where': 'This computer',
 'claim_id': 'CLM-1007',
 'decision': 'refer',
 'payout': 0.0,
 'rationale': "The claim type is 'baggage'. The summary describes a lost item (phone dropped in a hotel) and the documents confirm the scenario occurred within the policy's bag coverage period. According to the Policy Record, 'Baggage' does not include expenses taken off by hotels. While the customer stated the phone was paid for, it must first be verified if the reimbursement request includes that payment or is solely for the replacement. However, the primary issue preventing an immediate decision is the **missing Repair Shop Report**. The claim requires a repair estimate to determine the loss value accurately (which is $0 in this case due to total loss) and is necessary to establish if the repair was done within the 3-hour waiting period or to provide documentation that the incident happened. Without the repair report, the insurer cannot verify the extent of the damage 

Now the 20 claims on each model. Each call runs on your own computer, so the time depends on your hardware.

In [24]:
qwen = run_locally("qwen3.5:0.8b", claims, labels)

qwen3.5:0.8b:   0%|          | 0/4 [00:00<?, ?it/s]

qwen3.5:0.8b: 1 of 4 decisions right, 0 with the payout too, in 6 seconds


In [25]:
gemma = run_locally("gemma3:1b", claims, labels)

gemma3:1b:   0%|          | 0/4 [00:00<?, ?it/s]

gemma3:1b: 2 of 4 decisions right, 0 with the payout too, in 24 seconds


In [26]:
granite = run_locally("granite4:350m", claims, labels)

granite4:350m:   0%|          | 0/4 [00:00<?, ?it/s]

No answer on CLM-1021: ValidationError
granite4:350m: 3 of 4 decisions right, 1 with the payout too, in 15 seconds


## Part 4: Accuracy, speed and memory side by side

One row for each model. Accuracy is the percentage of the 20 claims it got right, first on the decision alone and then on the decision and the payout together. Memory is what Ollama used on your computer, so it's blank for `openrouter/free`.

In [27]:
experiments = [qwen, gemma, granite]

rows = []
for results in experiments:
    count = len(results)
    rows.append({
        "model": results[0]["model"],
        "where": results[0]["where"],
        "decision right (%)": round(100 * sum(result["decision_right"] for result in results) / count),
        "decision and payout right (%)": round(100 * sum(result["right"] for result in results) / count),
        "seconds a claim": round(sum(result["seconds"] for result in results) / count, 1),
        "memory (GB)": results[0].get("memory"),
    })

summary = pd.DataFrame(rows)
summary

,model,where,decision right (%),decision and payout right (%),seconds a claim,memory (GB)
0,qwen3.5:0.8b,This computer,25,0,1.4,0.6
1,gemma3:1b,This computer,50,0,6.1,1.0
2,granite4:350m,This computer,75,25,3.7,1.0


### Accuracy against speed

The best place to be is the top left: accurate and fast. The local times depend on your computer.

In [28]:
fig = px.scatter(
    summary,
    x="seconds a claim",
    y="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy against speed",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### Accuracy against memory

This one has the local models only. Each of them uses under 2 GB of memory.

In [29]:
local = summary[summary["where"] == "This computer"]

fig = px.scatter(
    local,
    x="memory (GB)",
    y="decision and payout right (%)",
    text="model",
    title="Accuracy against memory",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### All three at once

The local models again, with memory, speed and accuracy. Drag the chart to turn it.

In [30]:
fig = px.scatter_3d(
    local,
    x="memory (GB)",
    y="seconds a claim",
    z="decision and payout right (%)",
    text="model",
    title="Accuracy, speed and memory",
    height=700,
)
fig.show()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Weighing up accuracy, cost and latency for a business task is a crucial
            activity for an AI Solutions Architect and AI FDE.<br/><br/>
            There will also be situations where your data needs to stay on your customer's hardware.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: Try finding new ways to shorten the prompt that preserves the accuracy.
            Try finding a new way to get high accuracy results with as few tokens as possible.<br/><br/>
            Also try experimenting with more models on OpenRouter and locally.<br/><br/>
            Stretch challenge: Now go back to the product from yesterday and change it to use local models via Ollama!<br/><br/>
            Stretch stretch challenge: Investigate renting a GPU with RunPod, running a vLLM server, and running the product
            from that! Claude Code can give you instructions if you're interested..<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Solution - assignment challenge

**Was this experiment run? No.** OpenRouter's daily free quota on this account is exhausted, so the network cells below were not executed. The local cells are runnable with the models already installed through Ollama. Every API cell catches errors and prints why it was skipped.

### 1. Shorten the prompt without losing the rules

`make_messages` sends the opening, five sections of the short wording, the full task text and compact JSON for every claim. Two safe savings:

* `INSTRUCTIONS` restates the three decisions at length. The `ClaimDecision` schema already asks for a rationale, so one sentence per decision is enough.
* The `Claims` section has a 90-day filing rule that cannot change these decisions, as well as a missing-document rule and three definitions that can. We keep only those in one line.

Everything that can change a decision or payout stays: `Limits`, the benefit section, `Add-ons` when present, `Never covered`, `Payment` and the missing-document rule. `make_user_prompt` is unchanged: the claim facts and policy record are not padding. The first cell measures the reduction locally, without API calls.

In [ ]:
COMPACT_INSTRUCTIONS = """Decide the claim from the wording and the policy record.
approve = covered and the required documents are attached; work out the payout from the Limits and Payment sections.
decline = not covered. refer = a required document is missing, the story and documents disagree, or the wording does not settle the question.
Give a short rationale naming the rule you used."""

# The Claims section has one decision rule and three definitions. The 90-day
# filing rule cannot change any of these 20 decisions, so we keep only the rest.
CLAIMS_RULES = """Required documents: if one is missing, refer the claim. A family member is a spouse or partner, child, parent, sibling, grandparent or grandchild. A physician is a licensed doctor who is not the traveler, a companion or a family member. A traveling companion is a traveler named on the policy record."""


def make_compact_wording(claim):
    """Keep every rule that can change a decision or a payout, drop repetitive prose."""
    headings = ["Limits", BENEFITS[claim["claim_type"]], "Never covered", "Payment"]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    return "All amounts are US dollars.\n\n" + "\n\n".join(sections[heading].strip() for heading in headings) + "\n\n" + CLAIMS_RULES


def make_compact_messages(claim):
    """A shorter prompt that keeps the same task and the same facts."""
    return [
        {"role": "system", "content": COMPACT_INSTRUCTIONS + "\n" + make_compact_wording(claim)},
        {"role": "user", "content": make_user_prompt(claim)},  # unchanged: the facts must stay
    ]


def prompt_size(messages):
    text = "\n".join(message["content"] for message in messages)
    return len(text), len(text.split())


sizes = pd.DataFrame([
    {"claim": claim["claim_id"],
     "baseline words": prompt_size(make_messages(claim))[1],
     "compact words": prompt_size(make_compact_messages(claim))[1]}
    for claim in claims
])
print(f"Words sent per claim: {sizes['baseline words'].mean():.0f} baseline vs {sizes['compact words'].mean():.0f} compact")
sizes.head()

### 2. Check that accuracy survives

`decide_locally_with` below is a copy of `decide_locally` that accepts our messages, so the local A/B test works even while the free quota is exhausted. Run it on a few claims on one local model: compare decisions right and payouts right for the two prompts. Then, when the quota resets, `run_ab_free` does the same on `openrouter/free` with a fixed named model if you want a like-for-like comparison (free routing may answer each call with a different model, which adds noise).

In [ ]:
MODEL = "openrouter/free"
CHECK_CLAIMS = claims[: min(4, len(claims))]


def decide_with_messages(claim, messages, model):
    """Same as decide(), but with the messages we choose."""
    start = time.time()
    response = ask(model, messages)
    answer = response.choices[0].message.parsed
    return {
        "model": model, "where": "OpenRouter", "claim_id": claim["claim_id"],
        "decision": answer.decision, "payout": answer.payout, "rationale": answer.rationale,
        "answered_by": response.model, "input_tokens": response.usage.prompt_tokens,
        "seconds": round(time.time() - start, 1),
    }


def run_ab_free(check_claims, model):
    rows = []
    for name, build in [("baseline", make_messages), ("compact", make_compact_messages)]:
        answers = [decide_with_messages(claim, build(claim), model) for claim in check_claims]
        scored = [score(answer, labels[answer["claim_id"]]) for answer in answers]
        rows.append({
            "prompt": name,
            "input tokens": round(sum(r["input_tokens"] for r in scored) / len(scored)),
            "decisions right": sum(r["decision_right"] for r in scored),
            "with payout right": sum(r["right"] for r in scored),
            "seconds": round(sum(r["seconds"] for r in scored)),
        })
    return pd.DataFrame(rows)


try:
    print(run_ab_free(CHECK_CLAIMS, MODEL).to_string(index=False))
except Exception as error:
    print(f"Not run: {type(error).__name__}: {error}")

The local test below needs no quota. It mirrors `decide_locally` (including the 300-token reply cap used by this CPU notebook) and scores the two prompts on two models. Use `TINY = True` claims and allow a few minutes on a CPU.

In [ ]:
def decide_locally_with(claim, messages, model):
    """decide_locally with our messages; a bad reply is recorded as no answer."""
    start = time.time()
    try:
        response = ollama.chat(
            model=model, messages=messages,
            format=ClaimDecision.model_json_schema(), think=False,
            options={"num_ctx": 8192, "num_predict": 300},
        )
        answer = ClaimDecision.model_validate_json(response.message.content)
        decision, payout = answer.decision, answer.payout
    except (ollama.ResponseError, ValidationError) as error:
        print(f"No answer on {claim['claim_id']}: {type(error).__name__}")
        decision, payout = "no answer", 0.0
    return {
        "model": model, "where": "This computer", "claim_id": claim["claim_id"],
        "decision": decision, "payout": payout,
        "memory": memory_in_use(model), "seconds": round(time.time() - start, 1),
    }


def run_ab_local(model, check_claims):
    rows = []
    for name, build in [("baseline", make_messages), ("compact", make_compact_messages)]:
        answers = [decide_locally_with(claim, build(claim), model) for claim in check_claims]
        scored = [score(answer, labels[answer["claim_id"]]) for answer in answers]
        rows.append({
            "prompt": name, "model": model,
            "decisions right": sum(r["decision_right"] for r in scored),
            "with payout right": sum(r["right"] for r in scored),
            "seconds a claim": round(sum(r["seconds"] for r in scored) / len(scored), 1),
        })
    return pd.DataFrame(rows)


local_ab = []
for model in ["qwen3.5:0.8b", "gemma3:1b"]:
    try:
        local_ab.append(run_ab_local(model, CHECK_CLAIMS))
    except Exception as error:
        print(f"{model}: not run ({type(error).__name__}: {error})")
if local_ab:
    print(pd.concat(local_ab, ignore_index=True).to_string(index=False))

### 3. More models

OpenRouter ids ending in `:free` are the ones worth trying, but the list changes and only some support Structured Outputs, which `ask` requires. Check openrouter.ai/models, then edit `MORE_MODELS`; each run is guarded so a model that is busy or unavailable does not stop the others. `openrouter/free` is useful for breadth, but it picks a different model for each call, so compare it with a named free model if you want a fair accuracy test.

Locally, `ollama list` shows what is installed and `ollama.pull` adds more. The CPU notebook keeps to models of 1B parameters or fewer so they run without a GPU.

In [ ]:
MORE_MODELS = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "google/gemma-3-27b-it:free",
]
# The ids above are examples: replace them with ids you find on openrouter.ai/models.
more_results = {}
for model in MORE_MODELS:
    try:
        more_results[model] = run(model, claims, labels)
    except Exception as error:
        print(f"{model}: not run ({type(error).__name__}: {error})")

# Locally: print(ollama.list()) to see installed models, then run any of them.
installed = []  # e.g. ["qwen3.5:0.8b", "gemma3:1b"]
for model in installed:
    try:
        more_results[model] = run_locally(model, claims, labels)
    except Exception as error:
        print(f"{model}: not run ({type(error).__name__}: {error})")

for model, results in more_results.items():
    count = len(results)
    print(f"{model}: {sum(r['decision_right'] for r in results)}/{count} decisions right, "
          f"{sum(r['right'] for r in results)}/{count} with payout, "
          f"{round(sum(r['seconds'] for r in results) / count, 1)}s a claim")